# Linear drag (Taylor 2.1-2.2)

*Class notebook, PHY 317.* Run each cell with Shift-Enter.

Drag on a sphere of diameter $D$ at speed $v$ is $f = bv + cv^2$ with $b = \beta D$, $c = \gamma D^2$ (air: $\beta = 1.6\times10^{-4}$ N s/m$^2$, $\gamma = 0.25$ N s$^2$/m$^4$). Which term matters is set by
$$\frac{f_{\rm quad}}{f_{\rm lin}} = \frac{\gamma}{\beta}\,D v \approx (1.6\times10^3\ \mathrm{s/m^2})\, D v .$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

g = 9.8
beta, gamma = 1.6e-4, 0.25   # air at STP

In [ ]:
for name, D, v in [("baseball", 0.07, 5), ("raindrop", 1e-3, 0.6), ("oil drop", 1.5e-6, 5e-5)]:
    print(f"{name:9s}  f_quad/f_lin = {gamma/beta * D * v:.1e}")

## Linear drag, solved

With $y$ up, $\tau = m/b$ and $v_{\rm ter} = mg/b$:
$$v_x = v_{x0}e^{-t/\tau}, \qquad x = v_{x0}\tau(1 - e^{-t/\tau}),$$
$$v_y = -v_{\rm ter} + (v_{y0} + v_{\rm ter})e^{-t/\tau}, \qquad y = -v_{\rm ter}t + (v_{y0} + v_{\rm ter})\tau(1 - e^{-t/\tau}).$$
Things to look for as you move $\tau$: $v_x$ drops to 37% after one $\tau$; $x$ never passes $v_{x0}\tau$; $v_y$ is within 5% of $-v_{\rm ter}$ after $3\tau$.

In [ ]:
@interact(tau=(0.1, 10.0, 0.1), vx0=(0, 30, 1), vy0=(-20, 30, 1))
def linear_drag(tau=2.0, vx0=10, vy0=10):
    vter = g * tau
    t = np.linspace(0, 8, 500)
    e = np.exp(-t / tau)
    vx, x = vx0 * e, vx0 * tau * (1 - e)
    vy, y = -vter + (vy0 + vter) * e, -vter * t + (vy0 + vter) * tau * (1 - e)

    fig, ax = plt.subplots(2, 2, figsize=(9, 5.5))
    ax[0, 0].plot(t, vx); ax[0, 0].set_title("v_x(t)")
    ax[0, 1].plot(t, x);  ax[0, 1].axhline(vx0 * tau, ls=":", color="gray"); ax[0, 1].set_title("x(t)   dotted: v_x0 tau")
    ax[1, 0].plot(t, vy); ax[1, 0].axhline(-vter, ls=":", color="gray"); ax[1, 0].set_title("v_y(t)   dotted: -v_ter")
    ax[1, 1].plot(t, y);  ax[1, 1].set_title("y(t)")
    for a in ax.flat: a.grid(alpha=0.3)
    for a in ax[1]: a.set_xlabel("t (s)")
    plt.tight_layout(); plt.show()

## What is $\tau$ for a real drop?

Stokes's law: $b = 3\pi\eta D$, so $\tau = m/b$ with $m = \rho_{\rm obj}\,\pi D^3/6$. Taylor's examples: a Millikan oil drop ($D = 1.5\ \mu$m, $\rho = 840$ kg/m$^3$) and a 0.2 mm water drop.

In [ ]:
@interact(D_mm=(0.001, 2.0, 0.001), rho_obj=(500, 8000, 100), medium=["air", "water"])
def stokes(D_mm=0.2, rho_obj=1000, medium="air"):
    eta = {"air": 1.7e-5, "water": 1.0e-3}[medium]   # viscosity, Pa s
    D = D_mm * 1e-3
    m = rho_obj * np.pi * D**3 / 6
    b = 3 * np.pi * eta * D
    print(f"m = {m:.2e} kg    b = {b:.2e} kg/s    tau = {m/b:.2e} s    v_ter = {m*g/b:.2e} m/s")